In [73]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import os 
import joblib
os.chdir(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine')


In [74]:
df = pd.read_csv(r'D:\End-to-end-ML\Customer-Risk-Escalation-Engine\data\raw_data\support_tkts.csv')
df

,ticket_id,created_at,customer_id,customer_segment,channel,product_area,issue_type,priority,status,sla_plan,initial_message,agent_first_reply,resolution_summary,resolution_time_hours,reopened,customer_sentiment,csat_score,has_attachment,platform,region
0,TCKT_000001,2024-01-31T05:14:27,CUST_00861,individual,email,data_export,account_access,low,resolved,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,Reset account credentials and confirmed succes...,36.53,0,very_negative,1,0,android,EU
1,TCKT_000002,2024-10-20T06:15:49,CUST_00770,individual,in_app,billing,security_concern,medium,closed_no_action,standard,I noticed a suspicious login on my account.,We take security very seriously. Our team is r...,Ticket closed without further action after no ...,238.32,0,neutral,3,0,web,NaN
2,TCKT_000003,2024-06-18T21:35:54,CUST_02559,small_business,chat,api_integration,bug,low,in_progress,standard,The api integration feature is not saving my c...,Thanks for reporting this bug. We will look in...,NaN,NaN,0,neutral,3,0,android,MEA
3,TCKT_000004,2025-12-25T15:59:52,CUST_03557,education,chat,analytics_dashboard,account_access,medium,in_progress,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,positive,5,1,android,LATAM
4,TCKT_000005,2023-08-27T16:08:33,CUST_09556,enterprise,phone_transcript,login_auth,billing_problem,low,resolved,gold,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,61.32,0,very_negative,2,0,web,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,TCKT_099996,2023-05-16T06:23:50,CUST_06828,education,in_app,analytics_dashboard,billing_problem,high,resolved,standard,My invoice amount is incorrect compared to the...,Thanks for reaching out about the billing issu...,Adjusted the invoice and issued a refund where...,7.97,0,neutral,0,0,desktop_app,EU
99996,TCKT_099997,2025-03-10T04:46:24,CUST_06621,education,in_app,billing,bug,low,resolved,standard,I am seeing an error message when I click on b...,Thanks for reporting this bug. We will look in...,Patched the bug and deployed a fix to production.,16.48,0,negative,2,0,api_client,MEA
99997,TCKT_099998,2025-11-07T19:33:01,CUST_03719,non_profit,chat,billing,bug,low,resolved,standard,I am seeing an error message when I click on b...,Thanks for reporting this bug. We will look in...,Patched the bug and deployed a fix to production.,56.99,0,very_positive,5,0,web,LATAM
99998,TCKT_099999,2022-06-20T22:29:10,CUST_02290,small_business,email,analytics_dashboard,account_access,low,open,standard,I cannot log in; the system says my password i...,Sorry to hear you're having trouble accessing ...,NaN,NaN,0,neutral,0,1,api_client,NaN


In [75]:
df['reopened'].value_counts()

reopened
0    94955
1     5045
Name: count, dtype: int64

In [76]:
cond_sentiment = df['customer_sentiment'].isin(['negative', 'very_negative'])
cond_csat      = df['csat_score'] <= 2
cond_status    = df['status'].isin(['closed_no_action', 'open', 'on_hold'])

df['escalated'] = (
    cond_sentiment & 
    cond_csat      & 
    cond_status
).astype(int)

print("Target Distribution:")
print(df['escalated'].value_counts())
print(f"\nEscalation Rate: {df['escalated'].mean()*100:.2f}%")

print("\nSignal Verification:")
for col in ['customer_sentiment', 'csat_score', 'status', 'priority', 'resolution_time_hours']:
    if df[col].dtype == 'object':
        print(f"\n{col}:")
        print(df.groupby(col)['escalated'].mean().sort_values(ascending=False).round(3))
    else:
        corr = df[col].corr(df['escalated'])
        print(f"\n{col} correlation with escalated: {corr:.4f}")

Target Distribution:
escalated
0    89917
1    10083
Name: count, dtype: int64

Escalation Rate: 10.08%

Signal Verification:

customer_sentiment:
customer_sentiment
very_negative    0.301
negative         0.197
neutral          0.000
positive         0.000
very_positive    0.000
Name: escalated, dtype: float64

csat_score correlation with escalated: -0.2214

status:
status
open                0.336
on_hold             0.336
closed_no_action    0.332
in_progress         0.000
resolved            0.000
Name: escalated, dtype: float64

priority:
priority
urgent    0.106
medium    0.101
high      0.101
low       0.100
Name: escalated, dtype: float64

resolution_time_hours correlation with escalated: 0.4292


In [77]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 21 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   ticket_id              100000 non-null  object 
 1   created_at             100000 non-null  object 
 2   customer_id            100000 non-null  object 
 3   customer_segment       100000 non-null  object 
 4   channel                100000 non-null  object 
 5   product_area           100000 non-null  object 
 6   issue_type             100000 non-null  object 
 7   priority               100000 non-null  object 
 8   status                 100000 non-null  object 
 9   sla_plan               100000 non-null  object 
 10  initial_message        100000 non-null  object 
 11  agent_first_reply      100000 non-null  object 
 12  resolution_summary     60113 non-null   object 
 13  resolution_time_hours  60113 non-null   float64
 14  reopened               100000 non-nul

In [78]:
df['escalated'].value_counts()

escalated
0    89917
1    10083
Name: count, dtype: int64

In [79]:
col = ['ticket_id','customer_id','channel','region','platform','customer_sentiment','csat_score','status']

In [80]:
df = df.drop(col,axis=1)

In [81]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   created_at             100000 non-null  object 
 1   customer_segment       100000 non-null  object 
 2   product_area           100000 non-null  object 
 3   issue_type             100000 non-null  object 
 4   priority               100000 non-null  object 
 5   sla_plan               100000 non-null  object 
 6   initial_message        100000 non-null  object 
 7   agent_first_reply      100000 non-null  object 
 8   resolution_summary     60113 non-null   object 
 9   resolution_time_hours  60113 non-null   float64
 10  reopened               100000 non-null  int64  
 11  has_attachment         100000 non-null  int64  
 12  escalated              100000 non-null  int32  
dtypes: float64(1), int32(1), int64(2), object(9)
memory usage: 9.5+ MB


In [82]:
df['ticket_created_date'] = pd.to_datetime(
    df['created_at']
).dt.normalize()



In [83]:
df['ticket_created_date'] .dtype

dtype('<M8[ns]')

In [84]:
df['ticket_created_date']

0       2024-01-31
1       2024-10-20
2       2024-06-18
3       2025-12-25
4       2023-08-27
           ...    
99995   2023-05-16
99996   2025-03-10
99997   2025-11-07
99998   2022-06-20
99999   2022-02-27
Name: ticket_created_date, Length: 100000, dtype: datetime64[ns]

In [85]:
df['created_year'] = df['ticket_created_date'].dt.year
df['created_month'] = df['ticket_created_date'].dt.month_name()
df['weekday_num'] = df['ticket_created_date'].dt.dayofweek

In [86]:
df['resolution_summary'] = df['resolution_summary'].fillna('unresolved')
df['is_unresolved'] = df['resolution_time_hours'].isnull().astype(int)
df['resolution_time_hours'] = df['resolution_time_hours'].fillna(-1)

In [87]:
df = df.drop(columns=['created_at','ticket_created_date'],axis=1)

In [88]:
# Combine all text into one NLP input column
df['combined_text'] = (
    df['initial_message'].astype(str) + " " +
    df['agent_first_reply'].astype(str) + " " +
    df['resolution_summary'].astype(str)
)

#Pull out text separately before any encoding
text_data = df[['combined_text']].copy()

print("Text data shape:", text_data.shape)
print("\nSample combined text:")
print(text_data['combined_text'].iloc[0])

Text data shape: (100000, 1)

Sample combined text:
I cannot log in; the system says my password is incorrect. Sorry to hear you're having trouble accessing your account. We will look into this and get back to you within the next 48 hours. Reset account credentials and confirmed successful login.


In [89]:
# Remove all raw text columns from tabular path
df = df.drop(columns=[
    'initial_message',
    'agent_first_reply',
    'resolution_summary',
    'combined_text'
])

print("Tabular df shape after dropping text columns:", df.shape)
print("\nRemaining columns:")
print(df.columns.tolist())

Tabular df shape after dropping text columns: (100000, 13)

Remaining columns:
['customer_segment', 'product_area', 'issue_type', 'priority', 'sla_plan', 'resolution_time_hours', 'reopened', 'has_attachment', 'escalated', 'created_year', 'created_month', 'weekday_num', 'is_unresolved']


In [90]:
priority_map = {
    'low': 1, 'medium': 2, 'high': 3, 'urgent': 4
}
df['priority'] = df['priority'].map(priority_map)

In [91]:
# Word count of combined text
text_data['word_count'] = text_data['combined_text'].str.split().str.len()

print("Combined text word count stats:")
print(text_data['word_count'].describe())

print("\nAny empty texts:")
print((text_data['word_count'] == 0).sum())

Combined text word count stats:
count    100000.000000
mean         35.064770
std           4.892252
min          24.000000
25%          32.000000
50%          35.000000
75%          39.000000
max          48.000000
Name: word_count, dtype: float64

Any empty texts:
0


In [92]:
cat_cols = df.select_dtypes(include=['O']).columns.tolist()
print(f"Categorical Columns ({len(cat_cols)}):")
print(cat_cols)

Categorical Columns (5):
['customer_segment', 'product_area', 'issue_type', 'sla_plan', 'created_month']


In [94]:
le = LabelEncoder()
cat_cols = [
    'customer_segment', 'product_area', 'issue_type', 'sla_plan', 'created_month'
]
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

print("Encoding done ✅")
print(df.dtypes)

Encoding done ✅
customer_segment           int64
product_area               int64
issue_type                 int64
priority                   int64
sla_plan                   int32
resolution_time_hours    float64
reopened                   int64
has_attachment             int64
escalated                  int32
created_year               int32
created_month              int32
weekday_num                int32
is_unresolved              int32
dtype: object


In [95]:
text_data

,combined_text,word_count
0,I cannot log in; the system says my password i...,42
1,I noticed a suspicious login on my account. We...,36
2,The api integration feature is not saving my c...,30
3,I cannot log in; the system says my password i...,33
4,My invoice amount is incorrect compared to the...,43
...,...,...
99995,My invoice amount is incorrect compared to the...,40
99996,I am seeing an error message when I click on b...,40
99997,I am seeing an error message when I click on b...,40
99998,I cannot log in; the system says my password i...,36


In [96]:
df['escalated']

0        0
1        0
2        0
3        0
4        0
        ..
99995    0
99996    0
99997    0
99998    0
99999    0
Name: escalated, Length: 100000, dtype: int32

In [97]:
df

,customer_segment,product_area,issue_type,priority,sla_plan,resolution_time_hours,reopened,has_attachment,escalated,created_year,created_month,weekday_num,is_unresolved
0,2,3,0,1,2,36.53,0,0,0,2024,4,2,0
1,2,2,7,2,2,238.32,0,0,0,2024,10,6,0
2,4,1,2,1,2,-1.00,0,0,0,2024,6,1,1
3,0,0,0,2,2,-1.00,0,1,0,2025,2,3,1
4,1,4,1,1,0,61.32,0,0,0,2023,1,6,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,0,0,1,3,2,7.97,0,0,0,2023,8,1,0
99996,0,2,2,1,2,16.48,0,0,0,2025,7,0,0
99997,3,2,2,1,2,56.99,0,0,0,2025,9,4,0
99998,4,0,0,1,2,-1.00,0,1,0,2022,6,0,1


## Train_Test_Split

In [98]:
y = df['escalated']
X = df.drop(columns=[
    'escalated'
])

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (100000, 12)
y shape: (100000,)


In [99]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Spliting combined_text the same way
text_train, text_test = train_test_split(
    text_data,
    test_size=0.2,
    random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"y_train : {y_train.shape}")
print(f"y_test  : {y_test.shape}")

X_train : (75000, 12)
X_test  : (25000, 12)
y_train : (75000,)
y_test  : (25000,)


In [100]:
X.columns

Index(['customer_segment', 'product_area', 'issue_type', 'priority',
       'sla_plan', 'resolution_time_hours', 'reopened', 'has_attachment',
       'created_year', 'created_month', 'weekday_num', 'is_unresolved'],
      dtype='object')

In [101]:
scale_cols = [
    'customer_segment', 'product_area', 'issue_type',
       'sla_plan', 'resolution_time_hours',
       'has_attachment', 'created_year', 'created_month', 'weekday_num',
       'is_unresolved'
]

scaler = StandardScaler()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])

X_test[scale_cols] = scaler.transform(X_test[scale_cols])

In [102]:
# Saving Files and Scaler Object
X_train.to_csv('data/processed_data/X_train.csv', index=False)
X_test.to_csv('data/processed_data/X_test.csv',   index=False)
y_train.to_csv('data/processed_data/y_train.csv', index=False)
y_test.to_csv('data/processed_data/y_test.csv',   index=False)

text_train.to_csv('data/processed_data/text_train.csv', index=False)
text_test.to_csv('data/processed_data/text_test.csv',   index=False)

joblib.dump(scaler, 'models/saved_models/scaler.pkl')

print("All files saved ✅")

All files saved ✅
